In [ ]:
plt.figure(figsize=(16, 7))
plt.plot(model_output_m3s_logNSE.index, a, label = "Height [m]")

plt.xlabel("Date")
plt.ylabel("Height (m)")
plt.title("Height function")
plt.legend()
plt.grid(True)
plt.show()

np.set_printoptions(suppress=True)

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
import numpy as np
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
# import geopandas as gpd
import pandas as pd
import seaborn as sns
from scipy.stats import qmc
from ipywidgets import IntProgress
import math as math

# importeren van ewatercycle
import ewatercycle
import ewatercycle.models
import ewatercycle.forcing

from ewatercycle_discharge import DischargeLocal

shape_file_area = 7.629080e+03 # in km^2

In [ ]:
basin_name = "boven_suriname"

historical_start_date = "2019-01-01"
historical_end_date = "2024-12-31"
future_start_data = "2027-01-01"
future_end_data = "2099-12-31"

shapefile = Path.home() / "BEP-Julian" / "BEP-Julian" / "Suriname_Model" / "boven_suriname.shp"

forcing_route_CMIP = Path.home() / "BEP-Julian" / "BEP-Julian" / "Forcing" / "CMIP"

forcing_route = Path.home() / "BEP-Julian" / "BEP-Julian" / "Forcing" / "ERA5_SUR_2019_2024"/ "work" / "diagnostic" / "script" 

Eigen_model = Path.home() / "BEP-Julian" / "BEP-Julian" / "Suriname_Model" / "discharge_bmi"

In [ ]:
# # Option one: Generate CMIP data
# cmip_dataset =  {
#    'project': 'CMIP6',
#    'activity': 'ScenarioMIP',
#    'exp': 'ssp245',                                                           # veranderen per generatie CMIP data
#    'mip': 'day',
#    'dataset': 'MPI-ESM1-2-HR',
#    'ensemble': 'r1i1p1f1',
#    'institute': 'DKRZ',
#    'grid': 'gn'
# }

# cmip_historical =  {
#    'project': 'CMIP6',
#    'exp': 'historical',
#    'dataset': 'MPI-ESM1-2-HR',
#    "ensemble": 'r1i1p1f1',
#    'grid': 'gn'
# }

## verschillende datapunten. Historical is CMIP ssp245 voor 2019-2024.
# ssp126_dir = forcing_route_CMIP / "SSP126_26-99"
# ssp245_dir = forcing_route_CMIP / "SSP245_26-99"
# ssp585_dir = forcing_route_CMIP / "SSP585_26-99"
# historical = forcing_route_CMIP / "historical"

# CMIP_forcing = ewatercycle.forcing.sources["LumpedMakkinkForcing"].generate(
#    dataset=cmip_dataset,                                                       # veranderen per generatie CMIP data
#    start_time=historical_start_date+"T00:00:00Z",
#    end_time=historical_end_date+"T00:00:00Z",
#    shape=shapefile,
#    directory=historical,                                                       # veranderen per generatie CMIP data
# )

In [ ]:
# Option two: load generated data
# Load historical data
historic_location = forcing_route_CMIP / "historical" / "work" / "diagnostic" / "script" 
HIST = ewatercycle.forcing.sources["LumpedMakkinkForcing"].load(directory=historic_location)

# Load SSP126 data
ssp126_location = forcing_route_CMIP / "SSP126_26-99" / "work" / "diagnostic" / "script" 
SSP126 = ewatercycle.forcing.sources["LumpedMakkinkForcing"].load(directory=ssp126_location)

# Load SSP245 data
ssp245_location = forcing_route_CMIP / "SSP245_26-99" / "work" / "diagnostic" / "script" 
SSP245 = ewatercycle.forcing.sources["LumpedMakkinkForcing"].load(directory=ssp245_location)

# Load SSP585 data
ssp585_location = forcing_route_CMIP / "SSP585_26-99" / "work" / "diagnostic" / "script" 
SSP585 = ewatercycle.forcing.sources["LumpedMakkinkForcing"].load(directory=ssp585_location)

ERA5_forcing = ewatercycle.forcing.sources["LumpedMakkinkForcing"].load(directory=forcing_route)
evap = ERA5_forcing.to_xarray()["evspsblpot"] #units: kg m^-2 s^-1 = mm/s, gesampled per dag
prec = ERA5_forcing.to_xarray()["pr"] #units: kg m^-2 s^-1 = mm/s, gesampled per dag

In [ ]:
def level(Qin, Qout, ERA5_forcing, A, L0):
    # Qin is array met discharge van rivier in meer, gebaseerd op E en P en dagelijks [m^3/s]
    # Qout is een standaard baseflow uit het meer [m^3/s]
    # E is verdampingsdata van ERA5, dagelijks [mm/s]
    # A is oppervlakte meer [km^2]
    # L0 is een gegeven begin hoogte [m]
    # L0 moet handmatig worden ingevoerd omdat in de toekomst niet zeker is hoe hoog het meer gaat staan op het beginpunt van de forcing data
    
    dt = 3600*24
    E = ERA5_forcing.to_xarray()["evspsblpot"] /1000 #* dt
    L = np.zeros(len(Qin))
    L[0] = L0
    A = A * 10**6
    
    for i in range(len(Qin)-1):
        dL = ((Qin.iloc[i] - Qout) / A) - E[i]
        L[i+1] = L[i] + dL*dt 
        if L[i+1] > 48.5:
            L[i+1] = 48.5
    return L

In [ ]:
model = DischargeLocal(forcing = ERA5_forcing)
params = [1.30978424, 0.39171679, 1.05045276]
config_file, _ = model.setup(parameters=params, cfg_dir=Eigen_model)

model.initialize(config_file)
Q_m = []
time = []

while model.time < model.end_time:
    model.update()
    Q_m.append(model.get_value("Q")[0])
    time.append(pd.Timestamp(model.time_as_datetime))

model.finalize()
model_output_mmday = pd.Series(data=Q_m, name="Modelled discharge", index=time)
# print(model_output_mmday.max())
model_output_m3s_test = model_output_mmday * shape_file_area * 1000 / 86400

In [ ]:
forcing_list = [HIST, SSP126, SSP245, SSP585] 
output = []
years = []
params = [1.30978424, 0.39171679, 1.05045276]

for forcings in forcing_list:
    model = DischargeLocal(forcing=forcings)
    config_file, _ = model.setup(
        parameters=params, 
        cfg_dir = Eigen_model,
    )

    model.initialize(config_file)

    Q_m = []
    time = []
    
    while model.time < model.end_time:
        model.update()
        Q_m.append(model.get_value("Q")[0])
        time.append(pd.Timestamp(model.time_as_datetime))
        
    output.append(Q_m)
    years.append(time)
    
    del Q_m, time
    model.finalize()

In [ ]:
historical_output = pd.Series(data=output[0], name="Historical", index=years[0])["2019-01-01":]
SSP126_output = pd.Series(data=output[1], name="SSP126", index=years[1])["2027-01-01":]
SSP245_output = pd.Series(data=output[2], name="SSP245", index=years[2])["2027-01-01":]
SSP585_output = pd.Series(data=output[3], name="SSP585", index=years[3])["2027-01-01":]

# Convert mm/d to m3/s
factor = shape_file_area / 86.4
historical_output *= factor
SSP126_output *= factor
SSP245_output *= factor
SSP585_output *= factor

In [ ]:
parameter_test = [1.30978424, 0.39171679, 1.05045276]
model = DischargeLocal(forcing = ERA5_forcing)
config_file, _ = model.setup(parameters=parameter_test, cfg_dir=Eigen_model)

model.initialize(config_file)
Q_m = []
time = []

while model.time < model.end_time:
    model.update()
    Q_m.append(model.get_value("Q")[0])
    time.append(pd.Timestamp(model.time_as_datetime))

model.finalize()

In [ ]:
# plot historische data als test

plt.figure(figsize=(16, 7))
evap_hist = HIST.to_xarray()["evspsblpot"] #units: kg m^-2 s^-1 = mm/s, gesampled per dag
prec_hist = HIST.to_xarray()["pr"] #units: kg m^-2 s^-1 = mm/s, gesampled per dag
# prec_hist.plot()

In [ ]:
prec = prec.compute()
evap = evap.compute()

prec_hist = prec_hist.compute()
evap_hist = evap_hist.compute()
thr = 1e-7
era_wet = prec.values[prec.values > thr]

hist_wet = prec_hist.values[
    prec_hist.values > thr
]

q = np.linspace(0,1,1000)

era_q_pr = np.quantile(era_wet, q)

hist_q_pr = np.quantile(hist_wet, q)

era_q_evap = np.quantile(evap.values, q)

hist_q_evap = np.quantile(evap_hist.values,q)

In [ ]:
pr = prec_hist.values.copy()
wet_frac_era = (prec.values > thr).mean()
# find CMIP threshold that yields same wet-day fraction
thr_cmip = np.quantile( prec_hist.values, 1 - wet_frac_era)
mask = prec_hist.values > thr_cmip

pr_corr = np.zeros_like(pr)

pr_corr[mask] = np.interp(pr[mask], hist_q_pr, era_q_pr, left=era_q_pr[0], right=era_q_pr[-1])

prec_hist_corr = xr.DataArray(pr_corr, coords=prec_hist.coords, dims=prec_hist.dims, attrs=prec_hist.attrs,)

scale = (prec.mean()/prec_hist_corr.mean())**0.5

prec_hist_corr *= scale

In [ ]:
ev = evap_hist.values.copy()

ev_corr = np.interp(ev, hist_q_evap, era_q_evap, left=era_q_evap[0], right=era_q_evap[-1])

evap_hist_corr = xr.DataArray(ev_corr, coords=evap_hist.coords, dims=evap_hist.dims, attrs=evap_hist.attrs,)

In [ ]:
ssp126_ds = SSP126.to_xarray()

pr126 = ssp126_ds["pr"].compute()
evap126 = ssp126_ds["evspsblpot"].compute()

pr126_np = pr126.values.copy()
pr126_corr = np.zeros_like(pr126_np)

# use historical wet-day threshold
mask = pr126_np > thr_cmip

pr126_corr[mask] = np.interp(
    pr126_np[mask],
    hist_q_pr,
    era_q_pr,
    left=era_q_pr[0],
    right=era_q_pr[-1]
)

pr126_corr = xr.DataArray(
    pr126_corr,
    coords=pr126.coords,
    dims=pr126.dims,
    attrs=pr126.attrs,
)

# apply same scaling as historical
pr126_corr *= scale

evap126_corr = xr.DataArray(
    np.interp(
        evap126.values,
        hist_q_evap,
        era_q_evap,
        left=era_q_evap[0],
        right=era_q_evap[-1]
    ),
    coords=evap126.coords,
    dims=evap126.dims,
    attrs=evap126.attrs,
)

ssp245_ds = SSP245.to_xarray()

pr245 = ssp245_ds["pr"].compute()
evap245 = ssp245_ds["evspsblpot"].compute()

pr245_np = pr245.values.copy()
pr245_corr = np.zeros_like(pr245_np)

mask = pr245_np > thr_cmip

pr245_corr[mask] = np.interp(
    pr245_np[mask],
    hist_q_pr,
    era_q_pr,
    left=era_q_pr[0],
    right=era_q_pr[-1]
)

pr245_corr = xr.DataArray(
    pr245_corr,
    coords=pr245.coords,
    dims=pr245.dims,
    attrs=pr245.attrs,
)

pr245_corr *= scale

evap245_corr = xr.DataArray(
    np.interp(
        evap245.values,
        hist_q_evap,
        era_q_evap,
        left=era_q_evap[0],
        right=era_q_evap[-1]
    ),
    coords=evap245.coords,
    dims=evap245.dims,
    attrs=evap245.attrs,
)

ssp585_ds = SSP585.to_xarray()

pr585 = ssp585_ds["pr"].compute()
evap585 = ssp585_ds["evspsblpot"].compute()

pr585_np = pr585.values.copy()
pr585_corr = np.zeros_like(pr585_np)

mask = pr585_np > thr_cmip

pr585_corr[mask] = np.interp(
    pr585_np[mask],
    hist_q_pr,
    era_q_pr,
    left=era_q_pr[0],
    right=era_q_pr[-1]
)

pr585_corr = xr.DataArray(
    pr585_corr,
    coords=pr585.coords,
    dims=pr585.dims,
    attrs=pr585.attrs,
)

pr585_corr *= scale

evap585_corr = xr.DataArray(
    np.interp(
        evap585.values,
        hist_q_evap,
        era_q_evap,
        left=era_q_evap[0],
        right=era_q_evap[-1]
    ),
    coords=evap585.coords,
    dims=evap585.dims,
    attrs=evap585.attrs,
)

In [ ]:
SSP126_corr_ds = SSP126.to_xarray().copy()
SSP126_corr_ds["pr"] = xr.DataArray( pr126_corr, coords=pr126.coords, dims=pr126.dims, attrs=pr126.attrs, )
SSP126_corr_ds["evspsblpot"] = xr.DataArray( evap126_corr, coords=evap126.coords, dims=evap126.dims, attrs=evap126.attrs,)

SSP245_corr_ds = SSP245.to_xarray().copy()
SSP245_corr_ds["pr"] = xr.DataArray( pr245_corr, coords=pr245.coords, dims=pr245.dims, attrs=pr245.attrs, )
SSP245_corr_ds["evspsblpot"] = xr.DataArray( evap245_corr, coords=evap245.coords, dims=evap245.dims, attrs=evap245.attrs,)

SSP585_corr_ds = SSP585.to_xarray().copy()
SSP585_corr_ds["pr"] = xr.DataArray( pr585_corr, coords=pr585.coords, dims=pr585.dims, attrs=pr585.attrs, )
SSP585_corr_ds["evspsblpot"] = xr.DataArray( evap585_corr, coords=evap585.coords, dims=evap585.dims, attrs=evap585.attrs,)

HIST_corr_ds = HIST.to_xarray().copy()
HIST_corr_ds["pr"] = xr.DataArray( prec_hist_corr, coords=prec_hist.coords, dims=prec_hist.dims, attrs=prec_hist.attrs, )
HIST_corr_ds["evspsblpot"] = xr.DataArray( evap_hist_corr, coords=evap_hist.coords, dims=evap_hist.dims, attrs=evap_hist.attrs,)

In [ ]:
HIST_corr_ds.to_netcdf(
    historic_location / "forcing_biascorr.nc"
)
SSP126_corr_ds.to_netcdf(
    ssp126_location / "forcing_biascorr.nc"
)
SSP245_corr_ds.to_netcdf(
    ssp245_location / "forcing_biascorr.nc"
)
SSP585_corr_ds.to_netcdf(
    ssp585_location / "forcing_biascorr.nc"
)

HIST_corr = HIST.model_copy(deep=True)

SSP126_corr = SSP126.model_copy(deep=True)

SSP245_corr = SSP245.model_copy(deep=True)

SSP585_corr = SSP585.model_copy(deep=True)

In [ ]:
HIST_corr.filenames["pr"] = "forcing_biascorr.nc"
HIST_corr.filenames["evspsblpot"] = "forcing_biascorr.nc"

SSP126_corr.filenames["pr"] = "forcing_biascorr.nc"
SSP126_corr.filenames["evspsblpot"] = "forcing_biascorr.nc"

SSP245_corr.filenames["pr"] = "forcing_biascorr.nc"
SSP245_corr.filenames["evspsblpot"] = "forcing_biascorr.nc"

SSP585_corr.filenames["pr"] = "forcing_biascorr.nc"
SSP585_corr.filenames["evspsblpot"] = "forcing_biascorr.nc"

In [ ]:
test = HIST_corr.to_xarray()

In [ ]:
forcing_list = [HIST_corr, SSP126_corr, SSP245_corr, SSP585_corr] 
output_corr = []
years_corr = []
params = [1.30978424, 0.39171679, 1.05045276]

for forcings in forcing_list:
    model = DischargeLocal(forcing=forcings)
    config_file, _ = model.setup(
        parameters=params, 
        cfg_dir = Eigen_model,
    )

    model.initialize(config_file)

    Q_m = []
    time = []
    
    while model.time < model.end_time:
        model.update()
        Q_m.append(model.get_value("Q")[0])
        time.append(pd.Timestamp(model.time_as_datetime))
        
    output_corr.append(Q_m)
    years_corr.append(time)
    
    del Q_m, time
    model.finalize()

In [ ]:
historical_output = pd.Series(data=output_corr[0], name="Historical", index=years[0])["2019-01-01":]
SSP126_output = pd.Series(data=output_corr[1], name="SSP126", index=years[1])["2027-01-01":]
SSP245_output = pd.Series(data=output_corr[2], name="SSP245", index=years[2])["2027-01-01":]
SSP585_output = pd.Series(data=output_corr[3], name="SSP585", index=years[3])["2027-01-01":]

# Convert mm/d to m3/s
factor = shape_file_area / 86.4
historical_output *= factor
SSP126_output *= factor
SSP245_output *= factor
SSP585_output *= factor

In [ ]:
# quantile mapped CMIP neerslag data vergelijken met ERA5 en oude CMIP

def ecdf(x):
    x = np.asarray(x)
    x = x[np.isfinite(x)]      # remove NaNs
    x = np.sort(x)
    y = np.arange(1, len(x)+1) / len(x)
    return x, y

# x_cmip, y_cmip = ecdf(historical_output)
x_cmip, y_cmip = ecdf(test["pr"])
x_cmip_oud, y_cmip_oud = ecdf(prec_hist)
x_era5, y_era5 = ecdf(prec)

# x_era5 *= 3600*24

plt.figure(figsize=(16, 7))

plt.plot(x_cmip * 86400, y_cmip, label='CMIP gecorrigeerd')
plt.plot(x_cmip_oud * 86400, y_cmip_oud, label='CMIP')
plt.plot(x_era5 * 86400, y_era5, label='ERA5')

plt.xlabel('Neerslag [mm/d]')
plt.ylabel('Cumulatieve kans [-]')
plt.title("Regenval CMIP en ERA5")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
# quantile mapped CMIP verdamping data vergelijken met ERA5 en oude CMIP
x_cmipE, y_cmipE = ecdf(test["evspsblpot"])
x_cmip_oudE, y_cmip_oudE = ecdf(evap_hist)
x_era5E, y_era5E = ecdf(evap)

# x_era5 *= 3600*24

plt.figure(figsize=(16, 7))

plt.plot(x_cmipE, y_cmipE, label='CMIP gecorrigeerd')
plt.plot(x_cmip_oudE, y_cmip_oudE, label='CMIP')
plt.plot(x_era5E, y_era5E, label='ERA5')

plt.xlabel('Verdamping [mm/s]')
plt.ylabel('Cumulatieve kans [-]')
plt.title("Regenval CMIP en ERA5")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
def droogte(level, L_crit, start_date):

    # Create daily dates
    dates = pd.date_range(
        start=start_date,
        periods=len(level),
        freq="D"
    )

    # Make Series
    level = pd.Series(level, index=dates)

    # Determine drought days
    drought = level < L_crit

    # Daily deficit (0 if level >= L_crit)
    deficit = (L_crit - level).clip(lower=0)

    # Find consecutive drought periods
    groups = (drought != drought.shift()).cumsum()

    events = []

    for _, group in drought.groupby(groups):

        if group.iloc[0]:  # only drought groups

            start = group.index[0]
            end = group.index[-1]

            duration = (end - start).days + 1

            # Deficit during this event
            event_deficit = deficit.loc[start:end]

            max_deficit = event_deficit.max()
            cum_deficit = event_deficit.sum()
            mean_deficit = event_deficit.mean()

            events.append({
                "start": start,
                "end": end,
                "duration_days": duration,
                "max_deficit": max_deficit,
                "mean_deficit": mean_deficit,
                "cum_deficit": cum_deficit
            })

    events = pd.DataFrame(events)

    return drought, deficit, events

In [ ]:
Q_out = 200
A = 1020
L0 = 46.5
SSP126_level_200 = level(SSP126_output, Q_out, SSP126_corr, A, L0)
SSP245_level_200 = level(SSP245_output, Q_out, SSP245_corr, A, L0)
SSP585_level_200 = level(SSP585_output, Q_out, SSP585_corr, A, L0)

In [ ]:
# hoogte voorspellingen 3 klimaatscenario's

import matplotlib.dates as mdates
plt.figure(figsize=(16, 7))
plt.axhline(42.92, color = "r", linestyle='--', label = "Kritische hoogte; 42,92m")
plt.axhline(40.5, color = "r", label = "Zeer kritische hoogte; 40,5m")
plt.plot(SSP126_output.index, SSP126_level_200, label = "SSP126")
plt.plot(SSP245_output.index, SSP245_level_200, label = "SSP245")
plt.plot(SSP585_output.index, SSP585_level_200, label = "SSP585")
plt.legend()
# plt.xlim("2025-01-01", "2060-01-01")
plt.xlim(pd.Timestamp("2024-12-31"),
         pd.Timestamp("2060-01-01"))
plt.xticks(pd.date_range("2025", "2060", freq="5YS"))
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.ylim(25, 50)
plt.xlabel("Datum")
plt.ylabel("Hoogte meer [m]")
# plt.title("Klimaatscenario's met Qout = 200 m^3/s")
plt.title("Hoogtevoorspellingen 3 klimaatscenario's")
plt.grid()

In [ ]:
drought_ssp126, deficit_ssp126, events_ssp126 = droogte(SSP126_level_200, L_crit=42.92, start_date="2027-01-01")
drought_ssp245, deficit_ssp245, events_ssp245 = droogte(SSP245_level_200, L_crit=42.92, start_date="2027-01-01")
drought_ssp585, deficit_ssp585, events_ssp585 = droogte(SSP585_level_200, L_crit=42.92, start_date="2027-01-01")

In [ ]:
# hoogte tekort per jaar 3 klimaatscenario's

plt.figure(figsize=(16, 7))
drought_ssp126, deficit_ssp126, events_ssp126 = droogte(SSP126_level_200, L_crit=42.92, start_date="2027-01-01")
drought_ssp245, deficit_ssp245, events_ssp245 = droogte(SSP245_level_200, L_crit=42.92, start_date="2027-01-01")
drought_ssp585, deficit_ssp585, events_ssp585 = droogte(SSP585_level_200, L_crit=42.92, start_date="2027-01-01")

year_deficit_ssp126 = deficit_ssp126.groupby(deficit_ssp126.index.year).sum()
year_deficit_ssp245 = deficit_ssp245.groupby(deficit_ssp245.index.year).sum()
year_deficit_ssp585 = deficit_ssp585.groupby(deficit_ssp585.index.year).sum()

plt.plot(year_deficit_ssp126.index, year_deficit_ssp126, marker = "o", label = "SSP126")
plt.plot(year_deficit_ssp245.index, year_deficit_ssp245, marker = "o", label = "SSP245")
plt.plot(year_deficit_ssp585.index, year_deficit_ssp585, marker = "o", label = "SSP585")
# plt.axhline(1000, color = "r", linestyle='--', label = "kritische hoogte")
plt.xlim(2025, 2060)
plt.ylim(0, 25000)
plt.xlabel("Datum")
plt.ylabel("Hoogte tekort [m]")
plt.title("Hoogte tekort per jaar")
plt.legend()
plt.grid();

In [ ]:
# dagen onder kritische hoogte 3 klimaatscenario's

plt.figure(figsize=(16, 4))
drought_ssp126, deficit_ssp126, events_ssp126 = droogte(SSP126_level_200, L_crit=42.92, start_date="2027-01-01")
drought_ssp245, deficit_ssp245, events_ssp245 = droogte(SSP245_level_200, L_crit=42.92, start_date="2027-01-01")
drought_ssp585, deficit_ssp585, events_ssp585 = droogte(SSP585_level_200, L_crit=42.92, start_date="2027-01-01")

year_drought_ssp126 = drought_ssp126.groupby(drought_ssp126.index.year).sum()
year_drought_ssp245 = drought_ssp245.groupby(drought_ssp245.index.year).sum()
year_drought_ssp585 = drought_ssp585.groupby(drought_ssp585.index.year).sum()
# print(year_deficit_ssp126)
plt.plot(year_drought_ssp126.index, year_drought_ssp126, marker = "o", label = "SSP126")
plt.plot(year_drought_ssp245.index, year_drought_ssp245, marker = "o", label = "SSP245")
plt.plot(year_drought_ssp585.index, year_drought_ssp585, marker = "o", label = "SSP585")

plt.xlabel("Datum")
plt.ylabel("Dagen kritische hoogte")
plt.title("Dagen per jaar onder kritische hoogte 42,92m")
plt.xlim(2025, 2060)
plt.legend()
plt.grid();

In [ ]:
# dagen onder zeer kritische hoogte 3 klimaatscenario's
plt.figure(figsize=(16, 4))
drought_ssp126, deficit_ssp126, events_ssp126 = droogte(SSP126_level_200, L_crit=40.5, start_date="2027-01-01")
drought_ssp245, deficit_ssp245, events_ssp245 = droogte(SSP245_level_200, L_crit=40.5, start_date="2027-01-01")
drought_ssp585, deficit_ssp585, events_ssp585 = droogte(SSP585_level_200, L_crit=40.5, start_date="2027-01-01")

year_drought_ssp126 = drought_ssp126.groupby(drought_ssp126.index.year).sum()
year_drought_ssp245 = drought_ssp245.groupby(drought_ssp245.index.year).sum()
year_drought_ssp585 = drought_ssp585.groupby(drought_ssp585.index.year).sum()
# print(year_deficit_ssp126)
plt.plot(year_drought_ssp126.index, year_drought_ssp126, marker = "o", label = "SSP126")
plt.plot(year_drought_ssp245.index, year_drought_ssp245, marker = "o", label = "SSP245")
plt.plot(year_drought_ssp585.index, year_drought_ssp585, marker = "o", label = "SSP585")

plt.xlabel("Datum")
plt.ylabel("Dagen kritische hoogte")
plt.title("Dagen per jaar onder zeer kritische hoogte 40,5m")
plt.xlim(2025, 2060)
plt.legend()
plt.grid();